In [ ]:
from functools import lru_cache
from io import BytesIO
from zipfile import ZipFile, is_zipfile

import pandas as pd
import requests

from paths import paths, prepare_preprocessing
from preprocessing.src.patients import load_patients, london_weights
from preprocessing.src.threading import parallelize

In [ ]:
prepare_preprocessing()

In [ ]:
PATIENTS_DIR = paths.raw.health.patients
QOF_FILE = paths.preprocessing.health.qof

YEAR_FROM = 2021
YEAR_TO = 2026

QOF_URLS = {
    2021: 'https://files.digital.nhs.uk/AC/3C964F/QOF2021_v2.zip',
    2022: 'https://files.digital.nhs.uk/90/6F833F/QOF_2122_V2.zip',
    2023: 'https://files.digital.nhs.uk/32/40B931/QOF%202022-23%20Raw%20data%20.csv%20files.zip',
    2024: 'https://files.digital.nhs.uk/DA/975A29/QOF2324.zip',
    2025: 'https://files.digital.nhs.uk/95/4708D7/QOF2425.zip',
    2026: 'https://files.digital.nhs.uk/05/37C52B/QOF_2526_v2.zip',
}

In [ ]:
COLUMNS = {
    'AF001': 'AF',
    'CHD001': 'CHD',
    'DM017': 'DM',
    'HF001': 'HF',
    'HYP001': 'HYP',
    'OB002': 'OB',
    'PAD001': 'PAD',
    'STIA001': 'STIA'

    # 'AST001': 'AST',
    # 'CAN001': 'CAN',
    # 'CKD001': 'CKD',
    # 'COPD001': 'COPD',
    # 'DEM001': 'DEM',
    # 'EP001': 'EP',
    # 'MH001': 'MH',
    # 'PC001': 'PC',
    # 'RA001': 'RA',
    # 'SMOK001': 'SMOK',
    # 'THY001': 'THY'
}

In [ ]:
# Download the annual NHS releases, reusing existing ZIPs
for year in range(YEAR_FROM, YEAR_TO + 1):
    output = paths.raw.health.qof_files[year]

    if output.is_file():
        continue

    print(f'Downloading QOF {year - 1}/{year % 100:02d}...', flush = True)
    
    response = requests.get(QOF_URLS[year], timeout = 120)
    response.raise_for_status()

    if not is_zipfile(BytesIO(response.content)):
        raise ValueError(f'The QOF download for {year} is not a ZIP archive.')

    output.parent.mkdir(parents = True, exist_ok = True)
    output.write_bytes(response.content)

In [ ]:
# Prepare practice-level ratios from the original register counts
indicators = {condition: indicator for indicator, condition in COLUMNS.items()}

annual_tables = []
for year in range(YEAR_FROM, YEAR_TO + 1):
    with ZipFile(paths.raw.health.qof_files[year]) as archive:
        member = next(
            name for name in archive.namelist()
            if name.upper().startswith('PREVALENCE_') and name.lower().endswith('.csv')
        )

        with archive.open(member) as stream:
            annual = pd.read_csv(stream).rename(columns = {'REGISTER_SIZE': 'REGISTER'})

    # Use the total practice list
    totals = (
        annual.loc[annual.PATIENT_LIST_TYPE.eq('TOTAL')]
        .groupby('PRACTICE_CODE').PRACTICE_LIST_SIZE.first()
    )
    selected = annual.loc[annual.GROUP_CODE.isin(indicators)]

    # Use the full heart-failure register, excluding the LVSD subset
    if 'REGISTER_NAME' in selected.columns:
        selected = selected.loc[
            selected.GROUP_CODE.ne('HF') | selected.REGISTER_NAME.eq('Heart Failure Register')
        ]

    prepared = pd.DataFrame(
        {
            'orgcode_id': selected.PRACTICE_CODE,
            'year': year - 2000,
            'areaid_id': selected.GROUP_CODE.map(indicators),
            'ratio': selected.REGISTER / selected.PRACTICE_CODE.map(totals),
        }
    )
    annual_tables.append(prepared)

pd.concat(annual_tables, ignore_index = True).to_csv(QOF_FILE, index = False)

In [ ]:
# Read the QOF data and pivot it to have one row per practice and year
data = (
    pd.read_csv(QOF_FILE)
    .rename(columns = {'orgcode_id': 'practice_code'})
    .pivot_table(index = ['practice_code', 'year'], columns = 'areaid_id', values = 'ratio', aggfunc = 'first')
    .reset_index()
    .rename(columns = COLUMNS)
    .rename_axis(None, axis = 1)[['practice_code', 'year', *COLUMNS.values()]]
)

In [ ]:
# Compute per-practice LSOA weights and practice populations
@lru_cache(maxsize = None)
def load_patient_snapshot(year):
    df_pat = load_patients(year)
    df_pat.rename(columns = str.lower, inplace = True)

    totals = df_pat.groupby('practice_code', observed = True)['number_of_patients'].transform('sum')
    df_pat['weight'] = df_pat['number_of_patients'] / totals

    london = london_weights()
    df_weights = df_pat.loc[df_pat.lsoa_code.isin(london.LSOA21CD), ['practice_code', 'lsoa_code', 'weight']]

    practice_pop = df_pat.groupby('practice_code', observed = True)['number_of_patients'].sum().to_dict()

    return df_weights, practice_pop

In [ ]:
# Distribute condition cases to LSOAs using patient-weighted shares and re-aggregate
data['year'] = data['year'].apply(lambda y: f'20{int(y):02d}')
condition_cols = [c for c in data.columns if c not in ('practice_code', 'year')]

def split_conditions_to_lsoas(df_year, df_pat_weights):
    mult_cols = {c: df_year[c] * df_year['practice_pop'] for c in condition_cols}
    df_cases = df_year[['practice_code', 'practice_pop']].assign(**mult_cols)

    df_long = df_cases.melt(
        id_vars = ['practice_code', 'practice_pop'],
        value_vars = condition_cols,
        var_name = 'condition',
        value_name = 'cases_practice',
    ).dropna(subset = ['cases_practice'])

    df_lsoa = df_long.merge(df_pat_weights, on = 'practice_code', how = 'inner', copy = False)

    df_lsoa['cases'] = df_lsoa['cases_practice'] * df_lsoa['weight']
    df_lsoa['pop_covered'] = df_lsoa['practice_pop'] * df_lsoa['weight']

    return (
        df_lsoa.groupby(['lsoa_code', 'condition'], observed = True, sort = False)
        .agg(cases = ('cases', 'sum'), patient_count = ('pop_covered', 'sum'))
        .reset_index()
        .rename(columns = {'lsoa_code': 'LSOA21CD'})
    )

In [ ]:
# Map conditions to LSOAs for each year
def process_year(year):
    y = str(year)
    df_year = data[data['year'] == y].copy()

    if df_year.empty:
        cols = ['LSOA21CD', *condition_cols]

        return pd.DataFrame(columns = cols)

    df_weights, practice_pop = load_patient_snapshot(year)

    df_year['practice_pop'] = df_year['practice_code'].map(practice_pop)
    df_year = df_year.dropna(subset = ['practice_pop'])

    df_cases = split_conditions_to_lsoas(df_year, df_weights)
    df_cases['prevalence'] = df_cases['cases'] / df_cases['patient_count']

    return (
        df_cases.pivot(index = 'LSOA21CD', columns = 'condition', values = 'prevalence')
        .fillna(0)
        .reset_index()
        .rename_axis(None, axis = 1)
    )

results = parallelize(process_year, list(range(YEAR_FROM, YEAR_TO + 1)), max_workers = 6)

In [ ]:
for i, result in enumerate(results):
    year = YEAR_FROM + i
    result.to_csv(paths.preprocessing.health.annual[year], index = False)

In [ ]:
# Combine results and compute mean prevalence per LSOA
(
    pd.concat(results, ignore_index = True)
    .groupby('LSOA21CD', observed = True, sort = False)
    .mean(numeric_only = True)
    .reset_index()
    .to_csv(paths.preprocessing.health.prevalence, index = False)
)